In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession\
                    .builder\
                    .master('local[*]')\
                    .appName('homework5')\
                    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/29 17:14:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/07/29 17:14:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/07/29 17:14:52 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [ ]:
#q1
spark.version

'4.0.0'

In [3]:
#q2
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

--2025-07-29 17:15:31--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.164.82.160, 3.164.82.197, 3.164.82.112, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.164.82.160|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet’

yellow_tripdata_202 100%[===================>]  61.36M  19.5MB/s    in 3.5s    

2025-07-29 17:15:35 (17.7 MB/s) - ‘yellow_tripdata_2024-10.parquet’ saved [64346071/64346071]



In [20]:
df = spark.read.parquet('yellow_tripdata_2024-10.parquet')

df.repartition(4).write.parquet('yellow_2024-10', mode='overwrite')

In [29]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [21]:
#q2 answer: 25mb
!ls -lh ./yellow_2024-10/

total 188928
-rw-r--r--@ 1 shawn  staff     0B Jul 29 17:29 _SUCCESS
-rw-r--r--@ 1 shawn  staff    22M Jul 29 17:29 part-00000-d0644900-e064-4b45-9256-bc9244a33399-c000.snappy.parquet
-rw-r--r--@ 1 shawn  staff    22M Jul 29 17:29 part-00001-d0644900-e064-4b45-9256-bc9244a33399-c000.snappy.parquet
-rw-r--r--@ 1 shawn  staff    22M Jul 29 17:29 part-00002-d0644900-e064-4b45-9256-bc9244a33399-c000.snappy.parquet
-rw-r--r--@ 1 shawn  staff    22M Jul 29 17:29 part-00003-d0644900-e064-4b45-9256-bc9244a33399-c000.snappy.parquet


In [26]:
df.createOrReplaceTempView('hw5')

In [31]:
from pyspark.sql import functions as F
df.withColumn('pickup_date', F.date_format(F.col('tpep_pickup_datetime'), 'yyyy-MM-dd'))\
    .filter(F.col('pickup_date') == '2024-10-15')\
    .filter(F.col('pickup_date').isNotNull())\
    .count()

128893

In [28]:
spark.sql(""" 
SELECT count(*)
FROM hw5
WHERE cast(tpep_pickup_datetime as date) = '2024-10-15'
""").show()

+--------+
|count(1)|
+--------+
|  128893|
+--------+



In [38]:
df.withColumn('duration', F.timestamp_diff('hour', 'tpep_pickup_datetime', 'tpep_dropoff_datetime'))\
    .orderBy(F.desc('duration'))\
    .select('duration')\
    .show(1)

+--------+
|duration|
+--------+
|     162|
+--------+
only showing top 1 row


q5: Spark’s User Interface which shows the application's dashboard runs on which local port?

4040

In [40]:
zones = spark.read.parquet('zones')

In [56]:
from pyspark.sql.window import Window
df_result = df.join(zones, on = df.PULocationID == zones.LocationID, how='inner')\
    .filter(F.col('borough') != 'Unknown')\
    .withColumn('tripcount', F.count('*').over(Window.partitionBy('Zone')))\
    .select('Zone', 'tripcount')\
    .distinct()\
    .orderBy('tripcount')

In [58]:
df_result.show(truncate=False)

+---------------------------------------------+---------+
|Zone                                         |tripcount|
+---------------------------------------------+---------+
|Governor's Island/Ellis Island/Liberty Island|1        |
|Arden Heights                                |2        |
|Rikers Island                                |2        |
|Green-Wood Cemetery                          |3        |
|Jamaica Bay                                  |3        |
|Charleston/Tottenville                       |4        |
|Eltingville/Annadale/Prince's Bay            |4        |
|Rossville/Woodrow                            |4        |
|Port Richmond                                |4        |
|West Brighton                                |4        |
|Crotona Park                                 |6        |
|Great Kills                                  |6        |
|Heartland Village/Todt Hill                  |7        |
|Mariners Harbor                              |7        |
|Oakwood      

25/07/29 18:37:09 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 186074 ms exceeds timeout 120000 ms
25/07/29 18:37:09 WARN SparkContext: Killing executors is not supported by current scheduler.
25/07/29 18:37:11 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [50]:
zones.select('Zone').distinct().count()

262

In [57]:
df_result.explain(True)

== Parsed Logical Plan ==
'Sort ['tripcount ASC NULLS FIRST], true
+- Deduplicate [Zone#591, tripcount#732L]
   +- Project [Zone#591, tripcount#732L]
      +- Project [VendorID#297, tpep_pickup_datetime#298, tpep_dropoff_datetime#299, passenger_count#300L, trip_distance#301, RatecodeID#302L, store_and_fwd_flag#303, PULocationID#304, DOLocationID#305, payment_type#306L, fare_amount#307, extra#308, mta_tax#309, tip_amount#310, tolls_amount#311, improvement_surcharge#312, total_amount#313, congestion_surcharge#314, Airport_fee#315, LocationID#589, Borough#590, Zone#591, service_zone#592, tripcount#732L]
         +- Project [VendorID#297, tpep_pickup_datetime#298, tpep_dropoff_datetime#299, passenger_count#300L, trip_distance#301, RatecodeID#302L, store_and_fwd_flag#303, PULocationID#304, DOLocationID#305, payment_type#306L, fare_amount#307, extra#308, mta_tax#309, tip_amount#310, tolls_amount#311, improvement_surcharge#312, total_amount#313, congestion_surcharge#314, Airport_fee#315, Loca